In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup

import datetime

import pandas as pd

from selenium import webdriver
from time import sleep
import os
import re

from docx import Document


In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'KZ NBKA' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.3")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

#writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)




Running KZ NBKA Web Scraping Tool v.1.3


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

		 'profile.default_content_setting_values.automatic_downloads': 1}

chromeOptions.add_experimental_option("prefs",prefs)


driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

In [4]:
regdict={
        regulatorName+' 1': 'https://www.nationalbank.kz/en/news/uchetnaya-registraciya-platezhnoy-organizacii',    

        regulatorName+' 2': 'https://www.nationalbank.kz/en/page/reestr-platezhnyh-uslug', 

        regulatorName+' 3': 'https://www.nationalbank.kz/en/news/sistemy-elektronnyh-deneg-kazahstana', 

        regulatorName+' 4': 'https://www.nationalbank.kz/kz/news/register-of-participants', 

        regulatorName+' 5': 'https://www.nationalbank.kz/en/news/reestr-platezhnyh-sistem', 


        }

Typology ={

        regulatorName+' 1': 'Registry of payment organizations',    

        regulatorName+' 2': 'Register of Important Payment Service Providers', 

        regulatorName+' 3': 'Electronic money system register', 

        regulatorName+' 4': 'Register of participants (companies) of the special regulatory regime', 

        regulatorName+' 5': 'Payment systems register', 

        }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')






In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


# ----Extract zip ---
zip_pattern = re.compile(r"\b\d{6}\b")
def extract_zip(address: str):
    """Return the first 6-digit ZIP in the address, or '' if none found."""
    match = zip_pattern.search(address)
    return match.group(0) if match else ''

# ----Extract info ---
phone_pattern = re.compile(r"(?:t[eе]l\.?|тел\.?)[:\s]*([+\d()[\]\-\s\.]+)", flags=re.IGNORECASE)
email_pattern = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
website_pattern = re.compile(r"(https?://\S+|www\.[^\s;]+)")
ceo_pattern = re.compile(r"(?:CEO|Governor|Director|President|Chairman|Manager|Vice President)[:\s]*([^\n;]+)", flags=re.IGNORECASE)

def clean_phone(raw):
    return re.sub(r"[^\d+()\-]", "", raw).strip("-")

def extract_fields(text):
    text = text or ""

    phones = [clean_phone(m) for m in phone_pattern.findall(text)]
    emails = email_pattern.findall(text)
    websites = website_pattern.findall(text)
    ceo = ceo_pattern.search(text)
    ceo_val = ceo.group(1).strip() if ceo else ""

    # remove matched pieces to get residual address
    cleaned = text
    for patt in (phone_pattern, email_pattern, website_pattern, ceo_pattern):
        cleaned = patt.sub(" ", cleaned)
    address = re.sub(r"\s+", " ", cleaned).strip(" ;,")

    return pd.Series({
        "address": address,
        "phone": "; ".join(phones),
        "email": "; ".join(emails),
        "website": "; ".join(websites),
        "ceo": ceo_val,
    })

In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------

for index, reg in enumerate(regdict):
    print(f'[Start New Reg] -- Working with list { reg} --')
    driver.get(regdict[reg])
    sleep(2)  # wait for the page to load
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    host_url = 'https://www.nationalbank.kz'

    if reg != regulatorName+' 3' and reg != regulatorName+' 2':

        
        file_url = host_url + soup.find('div', class_='posts-files').find('a')['href']
        print(f'[INFO] -- Downloading file from {file_url} -- ')
        driver.get(file_url)
        sleep(5)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        if len(dl_files)>0 and not dl_files[0].endswith('.tmp') and not dl_files[0].endswith('.crdownload'):
            sleep(3)
            print('[INFO] -- Check the download file --')
        else:
            print('[INFO] -- Maybe the file link is error, Change to xlsx -- ')
            dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]

        if dl_files[0].endswith('.docx') or dl_files[0].endswith('.doc'):
            # Normally RegCode is 5
            print('[INFO] -- The file is a docx file, begin to extract table -- ')
            doc = Document(dl_files[0])
            tables = doc.tables
            print(f"[INFO] -- Number of tables: {len(tables)}  --")
            if len(tables) == 0:
                data = pd.DataFrame()
                print(data.shape)
            else:
                # Build DataFrame from the first table (extract cell.text for each cell)
                data = pd.DataFrame([[cell.text for cell in row.cells] for row in tables[0].rows])
                print(data.shape)
                data.columns = data.iloc[0]
                data = data[1:]
            mask = data[data.columns[0]].astype(str).str.strip().str.isdigit()
            valid_data = data[mask].reset_index(drop=True)
            valid_data = valid_data.loc[:, ~valid_data.columns.duplicated()]
            for date_included, name_,name_operator,interal_id, company_info_,comments in zip(
                valid_data[valid_data.columns[1]], 
                valid_data[valid_data.columns[2]],
                valid_data[valid_data.columns[3]],
                valid_data[valid_data.columns[4]], 
                valid_data[valid_data.columns[5]],
                valid_data[valid_data.columns[6]],
                ):
                if len(name_.lstrip()) > 2:
                    if 'excluded' in comments.lower():
                        pass
                        #print('--- Excluded Company ---')
                        #print(date_included, name_.lstrip())
                    else:
                        address_1=extract_fields(company_info_)['address'].split(';')[0].split(':')[0].split('Fax')[0].split('fax')[0].strip().replace('e-mail', '').replace('email', '').replace('telephone','')
                        phone_=extract_fields(company_info_)['phone']
                        email_=extract_fields(company_info_)['email']
                        website_=extract_fields(company_info_)['website']
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['Name'].append(name_.lstrip())
                        sqldict['InternalID_1'].append(interal_id.replace('\n',' ').lstrip())
                        sqldict['InternalID_1_type'].append("Business identification number of the payment system’s operator (if there is one)")
                        sqldict['Name - Mother Company'].append(name_operator.lstrip())
                        sqldict['RegulationDate'].append(date_included)
                        sqldict['Address_1'].append(address_1)
                        sqldict['Phone'].append(phone_)
                        sqldict['Email'].append(email_)
                        sqldict['Website'].append(website_)
                        sqldict['Zip'].append(extract_zip(address_1))
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[2])
                        sqldict['Cntry'].append('KZ')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict = bourange_same_length_array(sqldict)
        elif dl_files[0].endswith('.xlsx') or dl_files[0].endswith('.xls'):
            with pd.ExcelFile(dl_files[0]) as xlsx:
                print(f"[INFO] -- Number of sheets: {len(xlsx.sheet_names)}  --")
                print(f"[INFO] -- Number of sheets: {xlsx.sheet_names}  --")
                # Load the first sheet into a DataFrame
                data = pd.read_excel(xlsx, sheet_name=xlsx.sheet_names[0])
                data.columns = data.iloc[0]
                data = data[1:]

                if reg == regulatorName+' 1':
                    for date_included, registration_number, name_,BIN_, address_,comments in zip(
                                                                                                data[data.columns[1]], 
                                                                                                data[data.columns[2]],
                                                                                                data[data.columns[3]],
                                                                                                data[data.columns[5]], 
                                                                                                data[data.columns[6]],
                                                                                                data[data.columns[-1]],
                                                                                                ):
                        if pd.notna(name_):
                            if len(str(name_).lstrip()) > 2 and 'number' not in registration_number:
                                if 'excluded' in str(comments).lower() or 'removed' in str(comments).lower():
                                    pass
                                    #print('--- Excluded Company ---')
                                    #print(date_included, name_.lstrip())
                                else:
                                    #print(date_included, name_)
                                    sqldict['ListProcessDate'].append(processdate)
                                    sqldict['ListName'].append(Typology[reg])
                                    sqldict['Name'].append(name_.lstrip())
                                    sqldict['InternalID_1'].append(str(registration_number).lstrip())
                                    sqldict['InternalID_1_type'].append("Payment company's registration number")
                                    sqldict['InternalID_2'].append(str(BIN_).lstrip())
                                    sqldict['InternalID_2_type'].append("Business identification number of the payment organization")
                                    sqldict['RegulationDate'].append(date_included)
                                    sqldict['Address_1'].append(address_.lstrip())
                                    sqldict['Zip'].append(extract_zip(address_.lstrip()))
                                    sqldict['RegCtry'].append(reg.split()[0])
                                    sqldict['RegCode'].append(reg.split()[1])
                                    sqldict['ListCode'].append(reg.split()[2])
                                    sqldict['Cntry'].append('KZ')
                                    sqldict['RegulationType'].append('Regulated')
                                    sqldict = bourange_same_length_array(sqldict)
                elif reg == regulatorName+' 4':

                    for name_, BIN_, No_QRUB_BK in zip(
                                                                                                data[data.columns[1]], 
                                                                                                data[data.columns[2]],
                                                                                                data[data.columns[4]],
                                                                                                ):
                        if pd.notna(name_):
                            if len(str(name_).lstrip()) > 2 :

                                    sqldict['ListProcessDate'].append(processdate)
                                    sqldict['Name'].append(name_.lstrip())
                                    sqldict['InternalID_1'].append(str(BIN_).lstrip())
                                    sqldict['InternalID_1_type'].append("Business Identification Number")
                                    sqldict['InternalID_2'].append(str(No_QRUB_BK).lstrip())
                                    sqldict['InternalID_2_type'].append("No. QRUB BK")
                                    sqldict['RegCtry'].append(reg.split()[0])
                                    sqldict['RegCode'].append(reg.split()[1])
                                    sqldict['ListCode'].append(reg.split()[2])
                                    sqldict['ListName'].append(Typology[reg])
                                    sqldict['Cntry'].append('KZ')
                                    sqldict['RegulationType'].append('Regulated')
                                    sqldict = bourange_same_length_array(sqldict)        
                    
    
    else:
        tables = soup.find_all('table')
        table = tables[0]  
        if reg == regulatorName+' 3':
            df = pd.read_html(str(table))[0]
            df.columns = df.iloc[0]
            df = df[1:]
            for name_, operator_name in zip(
                                                        df[df.columns[1]], 
                                                        df[df.columns[3]],

                                                        ):
                if pd.notna(name_):
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['Name'].append(name_.lstrip())
                        sqldict['Name - Mother Company'].append(operator_name.lstrip())
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[2])
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['Cntry'].append('KZ')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict = bourange_same_length_array(sqldict)
        elif reg == regulatorName+' 2':
            df = pd.read_html(str(table))[0]
            df.columns = df.iloc[0]
            df = df[1:]
            for date_included, name_, BIN_, company_info_ in zip(
                                                        df[df.columns[1]], 
                                                        df[df.columns[2]], 
                                                        df[df.columns[3]],
                                                        df[df.columns[4]],
                                                        ):
                if pd.notna(name_):
                        address_1=extract_fields(company_info_)['address'].split(';')[0].split(':')[0].split('Fax')[0].split('fax')[0].strip().replace('e-mail', '').replace('email', '').replace('telephone','')
                        phone_=extract_fields(company_info_)['phone']
                        email_=extract_fields(company_info_)['email']
                        website_=extract_fields(company_info_)['website']
                        zip_ = address_1.split(',')[0]
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['Address_1'].append(address_1)
                        sqldict['Phone'].append(phone_)
                        sqldict['Email'].append(email_)
                        sqldict['Website'].append(website_)
                        sqldict['Zip'].append(zip_)
                        sqldict['Name'].append(name_.lstrip())
                        sqldict['InternalID_1'].append(str(BIN_).lstrip())
                        sqldict['InternalID_1_type'].append("Business identification number of the payment service provider")
                        sqldict['RegulationDate'].append(date_included)
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[2])
                        sqldict['Cntry'].append('KZ')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict = bourange_same_length_array(sqldict)
    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))
            
            

[Start New Reg] -- Working with list KZ NBKA 1 --
[INFO] -- Downloading file from https://www.nationalbank.kz/file/download/114356 -- 
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 3  --
[INFO] -- Number of sheets: ['Реестр', 'Парақ2', 'Парақ3']  --
[Start New Reg] -- Working with list KZ NBKA 2 --


C:\Users\wuj1\AppData\Local\Temp\3\ipykernel_10412\2959484798.py:166: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


[Start New Reg] -- Working with list KZ NBKA 3 --


C:\Users\wuj1\AppData\Local\Temp\3\ipykernel_10412\2959484798.py:147: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]


[Start New Reg] -- Working with list KZ NBKA 4 --
[INFO] -- Downloading file from https://www.nationalbank.kz/file/download/114366 -- 
[INFO] -- Check the download file --
[INFO] -- Number of sheets: 1  --
[INFO] -- Number of sheets: ['Реестр']  --
[Start New Reg] -- Working with list KZ NBKA 5 --
[INFO] -- Downloading file from https://www.nationalbank.kz/file/download/109750 -- 
[INFO] -- Check the download file --
[INFO] -- The file is a docx file, begin to extract table -- 
[INFO] -- Number of tables: 1  --
(29, 8)


In [7]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename,index=False)

driver.quit()

sleep(3)



In [8]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,"""WOOPPAY"" LLP",09-17-001,Payment company's registration number,120340004314,Business identification number of the payment ...,...,,,,,,,,,,
1,,,,,,"""TipTop Pay Kazakhstan"" LLP",02-17-003,Payment company's registration number,160240029779,Business identification number of the payment ...,...,,,,,,,,,,
2,,,,,,"""QIWI Kazakhstan""(КИВИ Казахстан)"" LLP",02-17-004,Payment company's registration number,60640010247,Business identification number of the payment ...,...,,,,,,,,,,
3,,,,,,"""PayPoint ПО"" LLP",02-17-005,Payment company's registration number,140940007077,Business identification number of the payment ...,...,,,,,,,,,,
4,,,,,,"""Allpay (APA)"" LLP",02-17-006,Payment company's registration number,130940007263,Business identification number of the payment ...,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216,,,,,,American Express,,Business identification number of the payment ...,,,...,,,American Express Limited,,,,,,,
217,,,,,,MIR National Payment System,Registration number 1147746831352,Business identification number of the payment ...,,,...,,,«National Payment Card System» JSC,,,,,,,
218,,,,,,Payment system Kaspi.kz,BIN 200840000951,Business identification number of the payment ...,,,...,,,LLC Kaspi Pay,,,,,,,
219,,,,,,UPT,936777,Business identification number of the payment ...,,,...,,,UPT Ödeme Hizmetleri \nve Elektronik Para A.Ş.,,,,,,,
